In [0]:
%sql
create catalog if not exists clinicalforge;
create schema if not exists clinicalforge.metadata;
create schema if not exists clinicalforge.bronze;
create schema if not exists clinicalforge.silver;
create schema if not exists clinicalforge.gold


In [0]:
%sql
-- =========================================================================
-- LEVEL 1: TENANCY & ENVIRONMENT CORE (Customer & Connection Details)
-- =========================================================================

-- 1. CUSTOMERS / TENANTS (e.g., Boston Health, ForgeCare West Network)
CREATE TABLE IF NOT EXISTS clinicalforge.metadata.Customers (
    CustomerID INT  NOT NULL,
    CustomerCode VARCHAR(20) NOT NULL UNIQUE, -- e.g., BOSHOSP, FORGECLIN
    CustomerName VARCHAR(150) NOT NULL,
    SubscriptionTier VARCHAR(30) NOT NULL,
    IsActive BOOLEAN  NOT NULL,
    CreatedDateTime TIMESTAMP  NOT NULL,
    CONSTRAINT PK_Customers PRIMARY KEY (CustomerID)
);

-- 2. PRODUCTS / DATA PRODUCTS (e.g., ClinicalAnalytics, RevenueCycle)
CREATE TABLE IF NOT EXISTS clinicalforge.metadata.Products (
    ProductID INT  NOT NULL,
    ProductCode VARCHAR(30) NOT NULL UNIQUE, -- e.g., CLIN_FORGE, FIN_FORGE
    ProductName VARCHAR(100) NOT NULL,
    ProductDescription VARCHAR(255),
    IsActive BOOLEAN NOT NULL,
    CONSTRAINT PK_Products PRIMARY KEY (ProductID)
);
----3. CustomerProduct Table ID----
CREATE TABLE IF NOT EXISTS clinicalforge.metadata.CustomerProduct (
    CustomerProductID INT PRIMARY KEY,
    CustomerID INT,
    ProductID INT,
    FOREIGN KEY (CustomerID) REFERENCES clinicalforge.metadata.Customers(CustomerID),
    FOREIGN KEY (ProductID) REFERENCES clinicalforge.metadata.Products(ProductID)
);

-- 4. CONNECTION DETAILS (Dynamic Secrets Routing & Connection Strings)
CREATE TABLE IF NOT EXISTS clinicalforge.metadata.ConnectionDetails (
    ConnectionID INT  NOT NULL,
    CustomerProductID INT,
    ConnectionName VARCHAR(100) NOT NULL UNIQUE, -- e.g., conn_sql_boshosp_prod
    TargetPlatform VARCHAR(50) NOT NULL,       -- AzureSQL, AWS_RDS, Oracle, Snowflake
    HostServer VARCHAR(255) NOT NULL,          -- Server endpoint URL
    DatabaseName VARCHAR(150) NOT NULL,
    KeyVaultSecretName VARCHAR(150) NOT NULL,  -- ADF reads password/conn string from this Secret Name
    IsActive BOOLEAN NOT NULL,
    CONSTRAINT PK_ConnectionDetails PRIMARY KEY (ConnectionID),
    FOREIGN KEY (CustomerProductID) REFERENCES clinicalforge.metadata.CustomerProduct(CustomerProductID)
);

In [0]:
%sql

-- =========================================================================
-- LEVEL 2: STREAMING CONFIGURATION (Event Hub / Kafka Details)
-- =========================================================================

-- 5. EVENT HUB / KAFKA INGESTION DETAILS
CREATE TABLE IF NOT EXISTS clinicalforge.metadata.EventHubDetails (
    EventHubID INT  NOT NULL,
    CustomerProductID INT NOT NULL,
    NamespaceName VARCHAR(150) NOT NULL,       -- e.g., evh-ns-clinicalforge-prod
    TopicName VARCHAR(150) NOT NULL UNIQUE,    -- e.g., clinicalforge.telemetry.patient-vitals
    ConsumerGroup VARCHAR(100)  NOT NULL,
    PartitionCount INT  NOT NULL,
    KeyVaultConnSecretStr VARCHAR(150) NOT NULL, -- Secret containing the SAS token string
    TargetDeltaTablePath VARCHAR(255) NOT NULL, -- Where Databricks writes the raw stream
    CONSTRAINT PK_EventHubDetails PRIMARY KEY (EventHubID),
    FOREIGN KEY (CustomerProductID) REFERENCES clinicalforge.metadata.CustomerProduct(CustomerProductID)
);

In [0]:
%sql
-- =========================================================================
-- LEVEL 3: OBJECT MAPPING LAYER (Tables & Operational Status)
-- =========================================================================

-- 6. TABLES LIST MAPPING (Source-to-Warehouse Registry)
CREATE TABLE IF NOT EXISTS clinicalforge.metadata.TablesList (
    TableID INT  NOT NULL,
    CustomerProductID INT NOT NULL,
    SourceConnectionID INT NOT NULL,
    SourceSchema VARCHAR(50) NOT NULL,
    SourceTableName VARCHAR(100) NOT NULL,
    WarehouseSchema VARCHAR(50) NOT NULL,      -- e.g., DW_Gold
    WarehouseTableName VARCHAR(100) NOT NULL,   -- e.g., DimPatients / FactEncounters
    ExtractionType VARCHAR(30),-- DEFAULT 'Incremental' NOT NULL, -- Full, Incremental, CDC, Stream
    WatermarkColumn VARCHAR(100), --- DEFAULT 'LastModifiedDateTime' NOT NULL,
    LastExtractWatermark TIMESTAMP,--- DEFAULT '1900-01-01 00:00:00' NOT NULL,
    DatabricksNotebookPath VARCHAR(255) NOT NULL,
    IsActive BOOLEAN, --DEFAULT TRUE NOT NULL,
    CONSTRAINT PK_TablesList PRIMARY KEY (TableID),
    FOREIGN KEY (CustomerProductID) REFERENCES clinicalforge.metadata.CustomerProduct(CustomerProductID),
    CONSTRAINT FK_Tables_Connections FOREIGN KEY (SourceConnectionID) REFERENCES clinicalforge.metadata.ConnectionDetails(ConnectionID)
);


In [0]:
%sql
-- =========================================================================
-- LEVEL 4: SCHEMAS & DICTIONARIES (Table Fields Mappings)
-- =========================================================================

-- 7. SOURCE TABLE FIELDS & DATA TYPES
CREATE TABLE IF NOT EXISTS clinicalforge.metadata.SourceTableFields (
    FieldID INT  NOT NULL,
    TableID INT NOT NULL,
    CustomerProductID INT,
    FieldName VARCHAR(100) NOT NULL,
    DataType VARCHAR(50) NOT NULL,              -- VARCHAR, INT, DATETIME2, DECIMAL
    IsPrimaryKey BOOLEAN, -- DEFAULT FALSE NOT NULL,
    IsSensitivePHI BOOLEAN, --- DEFAULT FALSE NOT NULL,     -- 1 = Triggers Databricks auto-hashing/anonymisation
    CONSTRAINT PK_SourceTableFields PRIMARY KEY (FieldID),
    CONSTRAINT FK_SourceFields_Tables FOREIGN KEY (TableID) REFERENCES clinicalforge.metadata.TablesList(TableID),
    FOREIGN KEY (CustomerProductID) REFERENCES clinicalforge.metadata.CustomerProduct(CustomerProductID)
);

-- 8. WAREHOUSE (DW) TABLE & TARGET DESIGNATIONS
CREATE TABLE IF NOT EXISTS clinicalforge.metadata.CustomerWarehouseTables (
    WarehouseTableID INT  NOT NULL,
    CustomerProductID INT NOT NULL,
    TargetSchema VARCHAR(50), -- DEFAULT 'Gold' NOT NULL,
    WarehouseTableName VARCHAR(100) NOT NULL UNIQUE,
    LoadMethod VARCHAR(30) NOT NULL,
    WatermarkColumn VARCHAR(100), --- DEFAULT 'LastModifiedDateTime' NOT NULL,
    IsActive BOOLEAN, --- DEFAULT TRUE NOT NULL,
    CreatedDateTime TIMESTAMP, ---(3) DEFAULT SYSUTCDATETIME() NOT NULL,
    CONSTRAINT PK_WarehouseOrchestration PRIMARY KEY (WarehouseTableID),
    FOREIGN KEY (CustomerProductID) REFERENCES clinicalforge.metadata.CustomerProduct(CustomerProductID)

    --CONSTRAINT CK_Meta_LoadMethod CHECK (LoadMethod IN ('FullRefresh', 'IncrementalWatermark', 'AppendStream')),
    --- FOREIGN KEY (CustomerProductID) REFERENCES clinicalforge.metadata.CustomerProduct(CustomerProductID)
);

In [0]:
%sql
-- USE DATABASE clinicalforge;
-- GO

-- Create the central pipeline run orchestration control table
CREATE TABLE IF NOT EXISTS clinicalforge.metadata.PipelineRun (
    RunID VARCHAR(50) NOT NULL,                -- Generated UUID string tracking the overall workflow run
    CustomerProductID INT NOT NULL,
    TableID INT NOT NULL,
    HealthClientID VARCHAR(20) NOT NULL,       -- Storing the CustomerCode identifier
    TargetTableName VARCHAR(150) NOT NULL,     -- Destination Bronze table name (e.g., BOSHOSP_Patients)
    StartDateTime TIMESTAMP NOT NULL,
    EndDateTime TIMESTAMP,
    RecordsIngested INT,
    RunStatus VARCHAR(30) NOT NULL,            -- In-Progress, Success, Failed
    ErrorMessage VARCHAR(8000),
    CONSTRAINT PK_PipelineRun PRIMARY KEY (RunID),
    FOREIGN KEY (CustomerProductID) REFERENCES clinicalforge.metadata.CustomerProduct(CustomerProductID),
    FOREIGN KEY (TableID) REFERENCES clinicalforge.metadata.TablesList(TableID)
);
